<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 1 — Pipeline de preprocesamiento de texto</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De texto crudo a términos limpios · NLTK + spaCy (es_core_news_sm)</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Construir el pipeline que convierte una noticia cruda en una lista de términos limpios, tomando **decisiones justificadas** en cada paso (no recetas memorizadas). El corpus que limpien aquí es el mismo que usarán en el **Lab 2** para construir su motor de búsqueda, así que las decisiones de hoy se arrastran al resto de la unidad.


## 0 · Entorno

Ejecuten una sola vez. Si están en Colab, descomenten la primera línea.

In [ ]:
# !pip install -q nltk spacy && python -m spacy download es_core_news_sm
import re, unicodedata, json
from collections import Counter
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
from nltk.stem import SnowballStemmer

import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

print('Entorno listo.')

## Corpus

Noticias chiapanecas (sintéticas) con ruido real: HTML, URLs, emojis, acentos faltantes, dobles espacios. **No editen el corpus.**

In [ ]:
corpus_crudo = [
    {"id": "d01", "titulo": "Lluvias provocan inundaciones en Tuxtla",
     "texto": "Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. "
              "Proteccion Civil pidio a la poblacion no cruzar las calles anegadas. Mas info en https://chiapasparalelo.com/nota1 ."},
    {"id": "d02", "titulo": "Crisis hidrica golpea la region",
     "texto": "La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. "
              "Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos."},
    {"id": "d03", "titulo": "Cafe de Chiapas rompe record de exportacion",
     "texto": "<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> "
              "Los productores de la Sierra celebran precios al alza."},
    {"id": "d04", "titulo": "Sequia afecta cultivos de maiz",
     "texto": "La sequia afecta gravemente los cultivos de maiz y frijol en la region fronteriza. "
              "Los agricultores reportan perdidas de hasta el 40% y piden apoyos al gobierno estatal."},
    {"id": "d05", "titulo": "Turismo crece en el Canon del Sumidero",
     "texto": "El Canon del Sumidero recibio mas de 200 mil visitantes durante la temporada. "
              "Los recorridos en lancha y el avistamiento de fauna son los principales atractivos del parque nacional. #Chiapas"},
    {"id": "d06", "titulo": "Sismo de magnitud 5.1 frente a las costas",
     "texto": "Un sismo de magnitud 5.1 se registro frente a las costas de Chiapas la madrugada del martes. "
              "No se reportaron danos ni victimas, informo el Servicio Sismologico Nacional."},
    {"id": "d07", "titulo": "UPCh inaugura laboratorio de IA",
     "texto": "La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial "
              "equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx ."},
    {"id": "d08", "titulo": "Repunta la produccion de cacao",
     "texto": "La produccion de cacao en la region del Soconusco repunto este año tras varios ciclos a la baja. "
              "Cooperativas locales apuestan por el chocolate artesanal de origen para mercados premium."},
    {"id": "d09", "titulo": "San Cristobal, destino cultural",
     "texto": "San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias "
              "atraen a viajeros de todo el mundo. La gastronomia y el textil artesanal son protagonistas."},
    {"id": "d10", "titulo": "Avanza obra de infraestructura carretera",
     "texto": "Avanza la rehabilitacion de la carretera que conecta Tuxtla con la costa. "
              "La obra busca reducir tiempos de traslado y mejorar la seguridad vial para miles de automovilistas."},
    {"id": "d11", "titulo": "Alertan por casos de dengue",
     "texto": "La Secretaria de Salud alerto por un repunte de casos de dengue en municipios de tierra caliente. "
              "Pide a la poblacion eliminar criaderos de mosco y acudir al medico ante fiebre alta. 🦟"},
    {"id": "d12", "titulo": "Feria celebra el cafe y el cacao",
     "texto": "La feria regional celebro el cafe y el cacao chiapaneco con catas, musica y venta directa de productores. "
              "Miles de asistentes recorrieron los stands durante el fin de semana."},
    {"id": "d13", "titulo": "Restablecen servicio de agua potable",
     "texto": "El servicio de agua potable se restablecera de forma escalonada en las colonias afectadas por la falla en la red. "
              "El organismo operador pidio a los usuarios almacenar agua de manera responsable."},
    {"id": "d14", "titulo": "Estudiantes ganan concurso de robotica",
     "texto": "Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un concurso nacional de robotica 🤖 "
              "con un brazo robotico de bajo costo. El equipo representara a Mexico en una competencia internacional."},
]
print(f"{len(corpus_crudo)} documentos cargados.")

In [ ]:
df = pd.DataFrame(corpus_crudo)
df['n_chars'] = df['texto'].str.len()
df[['id', 'titulo', 'n_chars']]

## 1 · Cargar y explorar

Antes de limpiar, hay que **mirar** los datos. Una limpieza a ciegas destruye señal sin que se den cuenta.


**1.a** Estadísticas de longitud. Completen los cálculos.

In [ ]:
# TODO: reporten el numero de documentos, la longitud media en caracteres,
#       y la longitud media en PALABRAS (split ingenuo por espacios).
#       Pista: df['texto'].str.split().str.len()
raise NotImplementedError

**1.b** Detección de ruido. Les damos un detector de URLs ya hecho como ejemplo. **Completen** los detectores de etiquetas HTML y de emojis, y reporten en qué documentos aparece cada tipo de ruido.

In [ ]:
RE_URL  = re.compile(r'https?://\S+')
RE_HTML = None   # TODO: regex que capture etiquetas como <p> </b>
RE_EMOJI = None  # TODO: regex (o uso de unicodedata) para detectar emojis

for fila in corpus_crudo:
    urls = RE_URL.findall(fila['texto'])
    if urls:
        print(fila['id'], '-> URL:', urls)
    # TODO: imprimir tambien HTML y emojis encontrados por documento

> **Pregunta (defensa):** de los tres tipos de ruido, ¿cuál *podría* ser señal útil en algún dominio y por qué? Respondan en la celda de abajo.


_Su respuesta:_ 

## 2 · Tokenizar y normalizar

Tokenizar = decidir dónde empieza y termina cada unidad. Normalizar = que dos formas superficiales de la misma palabra colapsen en el mismo término.


**2.a** Comparen el `split` ingenuo contra un tokenizador real (spaCy). Observen las diferencias en puntuación y contracciones.

In [ ]:
ejemplo = corpus_crudo[0]['texto']

ingenuo = ejemplo.split()           # tokenizacion por espacios
print('Ingenuo  :', ingenuo[:12])

# TODO: tokenizar 'ejemplo' con spaCy (nlp) y mostrar los primeros 12 tokens.
#       Comenten al menos 2 diferencias concretas con el split ingenuo.
raise NotImplementedError

**2.b** Función de normalización. Debe: pasar a minúsculas, aplicar **Unicode NFC**, quitar HTML, URLs y colapsar espacios. **Decisión clave:** ¿quitan o conservan acentos? Impleméntenlo como un parámetro y justifiquen su elección por defecto más abajo.

In [ ]:
def normalizar(texto, quitar_acentos=False):
    """Devuelve el texto normalizado (aun como string, sin tokenizar)."""
    # TODO: minusculas + unicode NFC + limpiar HTML/URLs + colapsar espacios
    # TODO: si quitar_acentos=True, eliminar diacriticos (pista: unicodedata NFD + filtrar Mn)
    raise NotImplementedError

# prueba rapida
# print(normalizar(corpus_crudo[2]['texto']))

> **Decisión documentada:** ¿quitar acentos por defecto, sí o no? Den **un** argumento a favor y **uno** en contra, y digan cuál pesa más *para un buscador* donde el usuario casi nunca escribe acentos.


_Su decisión y justificación:_ 

## 3 · Stopwords con criterio

Filtrar palabras de función reduce ruido y dimensionalidad… pero la lista que viene "de fábrica" puede borrar señal de su dominio.


In [ ]:
stopwords_es = nlp.Defaults.stop_words
print('Total de stopwords de spaCy (es):', len(stopwords_es))
print(sorted(list(stopwords_es))[:30])

**3.a** Inspeccionen la lista y encuentren **al menos una** palabra que, en el dominio de noticias de Chiapas, *sí* podría ser señal (piensen en negaciones, intensificadores o términos que cambian el sentido). Construyan su propia lista `MIS_STOPWORDS` quitándola(s).

In [ ]:
CONSERVAR = set()   # TODO: palabras que NO quieren tratar como stopword (con criterio de dominio)
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR
# TODO: impriman cuantas quitaron y verifiquen que su decision tiene efecto

_Justifiquen qué conservaron y por qué (lo defenderán oralmente):_ 

## 4 · Stemming vs. lemmatización

Mismo objetivo, dos filosofías. Aquí **miden** la diferencia en vez de creerla.


In [ ]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    # TODO: normalizar -> tokenizar -> quitar stopwords/puntuacion -> aplicar stemmer
    raise NotImplementedError

def tokens_lemma(texto):
    # TODO: normalizar -> procesar con spaCy -> token.lemma_ -> quitar stopwords/puntuacion
    raise NotImplementedError

**4.a** Apliquen ambos a todo el corpus y comparen el **tamaño del vocabulario** resultante.

In [ ]:
# TODO: construir el vocabulario (set de terminos) con cada metodo sobre TODO el corpus
#       y reportar |V_stemming| vs |V_lemma|. Muestren 10 ejemplos donde difieran.
raise NotImplementedError

> **Decisión final:** ¿stemming o lemmatización para este corpus en español? Justifiquen con **los números** que acaban de obtener, no con la intuición.


_Su decisión y justificación:_ 

## 5 · Pipeline final + persistencia

Empaqueten su decisión en **una** función `preprocesar(texto) -> list[str]`. Esta función es el entregable más importante: el **Lab 2 la reutilizará tal cual** para indexar el corpus *y* para procesar las consultas. Si el corpus y las consultas no pasan por el mismo pipeline, su buscador no funcionará.


In [ ]:
def preprocesar(texto):
    """Pipeline definitivo del equipo: texto crudo -> lista de terminos limpios.
    Debe reflejar TODAS sus decisiones (acentos, stopwords, stemming/lemma)."""
    # TODO: implementar reutilizando lo que ya construyeron arriba
    raise NotImplementedError

# Demostracion
for fila in corpus_crudo[:3]:
    print(fila['id'], '->', preprocesar(fila['texto'])[:10])

In [ ]:
# Guardar el corpus crudo y el procesado para el Lab 2
corpus_procesado = [{'id': f['id'], 'titulo': f['titulo'],
                     'tokens': preprocesar(f['texto'])} for f in corpus_crudo]

with open('corpus_crudo.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_crudo, fh, ensure_ascii=False, indent=2)
with open('corpus_procesado.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_procesado, fh, ensure_ascii=False, indent=2)

print('Guardados: corpus_crudo.json y corpus_procesado.json')

## Entregables del Lab 1

- [ ] Celdas de exploración (1.a, 1.b) ejecutadas con sus salidas.
- [ ] `normalizar`, `tokens_stemming`, `tokens_lemma` y `preprocesar` implementadas.
- [ ] Las **tres decisiones justificadas** por escrito: acentos, stopwords, stemming/lemma.
- [ ] `corpus_procesado.json` generado y subido al repo (lo necesita el Lab 2).
- [ ] `AI_USAGE.md` actualizado si usaron IA.
